In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[2])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert
from sqlalchemy.orm import Session
import polars as pl
import re
from datetime import datetime

# Importando a partir do pacote 'app'
from app.config import db_engine
from app.models import GeneralSearch
from app.services import OLXService
from app.utils import read_scraping_targets_metadata

In [ ]:
# Base list for general search
general_data_rows = []

# Date
today = datetime.now()

# Regex
OLX_REGION_PATTERN = re.compile(r"https?://([^.]+)\.olx")

In [ ]:
async def scrap_olx():
    olx_service = OLXService()
    await search_for_general_links(olx_service=olx_service)
    await olx_service.terminate_client()

async def search_for_general_links(olx_service: OLXService):
    # Read the scraping targets metadata
    targets_metadata = read_scraping_targets_metadata().get("targets", [])
    itens = targets_metadata.items()

    for category, subcategories in itens:

        for subcategory, items in subcategories.items():
            append_subcategory = False
            if category in ["memory", "storage", "peripherals"]:
                append_subcategory = True
            if subcategory in ["motherboard"]:
                append_subcategory = True
            
            for item in items:
                item_name = item.get('pt', '')
                if append_subcategory or item_name == "":
                    item_name = subcategory.capitalize() + " " + item_name

                results = await olx_service.get_details_links(item_name)

                url = results.get('url', '')
                status = results.get('status', 400)
                links = results.get('links', [])
                
                if status != 200:
                    print(f'Erro no processamento da url: {url}')

                for link in links:
                    if link:
                        # Takes region
                        match = OLX_REGION_PATTERN.search(link)
                        region = match.group(1) if match else "unknown"

                        general_data_rows.append({
                            'category': category,
                            'subcategory': subcategory,
                            'item': item_name,
                            'url': url,
                            'status': status,
                            'region': region,
                            'link': link,
                            'datetime': today,
                        })

In [ ]:
await scrap_olx()

In [ ]:
# for store in stores:
#     store_name = store.get('name', '')
#     store_base_url = store.get('base_url', '')
    
#     print(f'Processing {store_name}')

    # if store_name == 'OLX':
    #     await scrap_olx(regions)

In [ ]:
# Creating dataframe
general_df = pl.DataFrame(general_data_rows)

if not general_df.is_empty():
    with Session(db_engine) as session:
        # Saving on sqlite
        session.execute(insert(GeneralSearch), general_df.to_dicts())
        session.commit()
else:
    print("No data found.")